# XGBoost challenger: submission without ratios/charging helpers

This notebook creates the Kaggle submission for the `no_ratios_charging` candidate. It preserves all files in `XGBoost/baseline/`, `XGBoost/advanced/`, and `XGBoost/ablation/`.

The candidate removes **12 engineered ratio/charging helper columns** but retains raw numeric inputs, frequency/digit features, ordinary and recipe-residual target encodings, and the recipe starting margin. The feature builders are copied exactly from the completed confirmation notebook.

**20 fits:** 10 stratified folds x 2 split seeds. Hyperparameters and both seed schedules are frozen to the completed confirmation run. Each fold predicts only its held-out rows for OOF evaluation, and predicts every test row for submission. Test probabilities are averaged equally across the 20 fits.

This exports the two-seed single-family challenger, **not** the old three-family rank blend. Confirmation OOF AUC was approximately **0.94580460** when averaging the two seeds; the leaderboard improvement is not guaranteed.

Select your existing `venv` kernel, **save this notebook**, then **Restart Kernel and Run All**. Training is left to you. No Optuna search, package installation, or Kaggle upload is performed.


## 1. Reproducible settings and input checks

No configuration edits are needed. Defaults match confirmation: CPU, five inner target-encoding folds, histogram bin count 512, 6,000 maximum trees, and 200-round early stopping.

Completed fits are checkpointed with both validation and test probabilities. Native XGBoost models are also saved by default. Interrupted runs can resume by restarting the kernel and running all cells again.

Save before running: checkpoint compatibility includes hashes of the saved notebook code, frozen configuration, CSV inputs, and library versions. Changed settings use a separate run directory. Do not run simultaneous writers to the same directory.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost as xgb
from IPython.display import display
from scipy.special import ndtr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

PIPELINE_VERSION = '1.0.0'
EXPERIMENT_NAME = 'no_ratios_charging'
GLOBAL_SEED = 20260915
N_SPLITS = 10
SPLIT_SEEDS = (20260932, 20261932)
INNER_TE_FOLDS = 5
MAX_BOOST_ROUNDS = 6000
EARLY_STOPPING_ROUNDS = 200
MAX_BIN = 512
DEVICE = 'cpu'
NTHREAD = -1
RESUME_COMPLETED_FOLDS = True
SAVE_MODELS = True

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 100)

assert N_SPLITS >= 2 and INNER_TE_FOLDS >= 2
assert MAX_BOOST_ROUNDS > EARLY_STOPPING_ROUNDS > 0
assert MAX_BIN >= 2 and SPLIT_SEEDS and len(set(SPLIT_SEEDS)) == len(SPLIT_SEEDS)
assert all(isinstance(seed, int) and 0 <= seed < 2**32 for seed in SPLIT_SEEDS)

VERSIONS = {
    'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
    'sklearn': sklearn.__version__, 'xgboost': xgb.__version__,
}
print('Versions:', VERSIONS)
print(f'{EXPERIMENT_NAME}: {N_SPLITS} folds x {len(SPLIT_SEEDS)} split seeds = '
      f'{N_SPLITS * len(SPLIT_SEEDS)} fits')


In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'playground-series-s6e9' / 'train.csv').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter in this repository or its XGBoost/challenger folder.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
STUDY_DIR = PROJECT_ROOT / 'XGBoost/challenger'
NOTEBOOK_PATH = STUDY_DIR / 'xgboost_no_ratios_charging_submission.ipynb'
CONFIG_PATH = STUDY_DIR / 'reference_configuration.json'
DATA_DIR = PROJECT_ROOT / 'playground-series-s6e9'
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_PATH = DATA_DIR / 'sample_submission.csv'
TARGET = 'Will_Buy_EV'
ID_COL = 'id'

with CONFIG_PATH.open(encoding='utf-8') as handle:
    frozen_configuration = json.load(handle)
confirmation_manifest = frozen_configuration['confirmation_manifest']
REFERENCE_SEARCH_PARAMS = dict(confirmation_manifest['reference_search_params'])
CANDIDATE_SPEC = dict(confirmation_manifest['experiments'][EXPERIMENT_NAME])

assert REFERENCE_SEARCH_PARAMS['scale_pos_weight'] == 1.0
assert CANDIDATE_SPEC == {
    'drop_groups': ['ratios', 'charging'],
    'te_mode': 'all', 'prior_scale': 1.0, 'smoothing_scale': 1.0,
    'description': 'Remove ratios and charging helpers; raw station counts stay.',
}
# Accidental changes to folds/seeds/budgets would no longer reproduce confirmation.
for key, value in {
    'global_seed': GLOBAL_SEED, 'n_splits': N_SPLITS,
    'split_seeds': list(SPLIT_SEEDS), 'inner_te_folds': INNER_TE_FOLDS,
    'max_boost_rounds': MAX_BOOST_ROUNDS, 'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'max_bin': MAX_BIN,
}.items():
    assert confirmation_manifest[key] == value, f'{key} differs from confirmation.'
if VERSIONS != confirmation_manifest['versions']:
    print('WARNING: library versions differ from confirmation; exact scores may change.')

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)
assert TARGET in train.columns and TARGET not in test.columns
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert train[ID_COL].notna().all() and test[ID_COL].notna().all()
assert train[TARGET].notna().all() and set(train[TARGET].unique()) == {'No', 'Yes'}
assert not set(train[ID_COL]).intersection(test[ID_COL])
assert sample_submission.columns.tolist() == [ID_COL, TARGET]
assert sample_submission[ID_COL].is_unique and sample_submission[ID_COL].notna().all()
assert len(sample_submission) == len(test)
assert set(sample_submission[ID_COL]) == set(test[ID_COL])

y = train[TARGET].map({'No': 0, 'Yes': 1}).astype(np.uint8).to_numpy()
TRAIN_IDS = train[ID_COL].to_numpy(copy=True)
TEST_IDS = test[ID_COL].to_numpy(copy=True)
RAW_FEATURES = [column for column in test.columns if column != ID_COL]
assert RAW_FEATURES == [column for column in train.columns if column not in (ID_COL, TARGET)]
assert np.bincount(y).min() >= max(N_SPLITS, INNER_TE_FOLDS) * 2

print(f'Train: {train.shape}; test: {test.shape}; positive rate: {y.mean():.6f}')
print('Frozen model parameters:', json.dumps(REFERENCE_SEARCH_PARAMS, indent=2))


## 2. Reference feature builders and candidate selection

The following feature builder, group registry, and cross-fitted encoders are copied from the successful confirmation notebook. No old notebook is imported or executed.

Train+test frequencies/vocabulary are label-free and deliberately match the confirmed pipeline. Any encoding using outcomes is fitted only on each outer training fold; its training encodings use five inner folds. Validation/test mappings never use validation/test outcomes.

The candidate keeps all target encodings and the recipe base margin. Only the 12 explicit ratio/charging helper columns are excluded; raw income, commute, car counts, and station counts remain.


In [ ]:
NUMERIC_COLS = [
    'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
    'Environmental_Concern_Level',
]
CATEGORICAL_COLS = [
    'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
    'Subsidy_Available', 'Range_Anxiety_Level',
]
assert set(NUMERIC_COLS + CATEGORICAL_COLS) == set(RAW_FEATURES)

train_medians = train[NUMERIC_COLS].median()
clip_bounds = {
    col: tuple(train[col].quantile([0.005, 0.995]).astype(float))
    for col in ('Annual_Income_USD', 'Daily_Commute_km')
}


def clean_raw(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[RAW_FEATURES].copy()
    for col in NUMERIC_COLS:
        out[col] = pd.to_numeric(out[col], errors='coerce').fillna(train_medians[col])
    for col in CATEGORICAL_COLS:
        out[col] = out[col].astype('string').fillna('__MISSING__').astype(str)
    return out


train_raw = clean_raw(train)
test_raw = clean_raw(test)
all_raw = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)

frequency_maps = {
    col: all_raw[col].value_counts(dropna=False, normalize=True)
    for col in RAW_FEATURES
}


def ev_recipe_score(frame: pd.DataFrame) -> np.ndarray:
    subsidy = frame['Subsidy_Available'].eq('Yes').to_numpy(dtype=np.float64)
    anxiety_penalty = frame['Range_Anxiety_Level'].map(
        {'Low': 0.0, 'Medium': -1.0, 'High': -3.0}
    ).fillna(-1.0).to_numpy(dtype=np.float64)
    return (
        1.2 * frame['Annual_Income_USD'].to_numpy(dtype=np.float64) / 100_000.0
        + 0.6 * frame['Environmental_Concern_Level'].to_numpy(dtype=np.float64)
        + 2.0 * subsidy
        + anxiety_penalty
    )


def ev_recipe_probability(frame: pd.DataFrame) -> np.ndarray:
    # The source rule is a probit-style threshold with unit Gaussian noise.
    return np.clip(ndtr(ev_recipe_score(frame) - 5.5), 1e-6, 1.0 - 1e-6)


def ev_recipe_logit_margin(frame: pd.DataFrame) -> np.ndarray:
    probability = ev_recipe_probability(frame)
    return np.log(probability / (1.0 - probability)).astype(np.float32)


def safe_divide(numerator: pd.Series, denominator: pd.Series, offset: float = 1.0) -> np.ndarray:
    return numerator.to_numpy(dtype=np.float64) / (denominator.to_numpy(dtype=np.float64) + offset)


def sanitize_feature_names(columns: list[str]) -> list[str]:
    counts: dict[str, int] = {}
    cleaned: list[str] = []
    for column in columns:
        base = re.sub(r'[^0-9A-Za-z_]+', '_', str(column)).strip('_') or 'feature'
        occurrence = counts.get(base, 0)
        counts[base] = occurrence + 1
        cleaned.append(base if occurrence == 0 else f'{base}_{occurrence}')
    return cleaned


def build_base_features(frame: pd.DataFrame) -> pd.DataFrame:
    x = frame[NUMERIC_COLS].astype(np.float32).copy()
    income = frame['Annual_Income_USD']
    commute = frame['Daily_Commute_km']
    cars = frame['Number_of_Cars_Owned']
    home_stations = frame['Charging_Stations_Near_Home']
    work_stations = frame['Charging_Stations_Near_Work']
    concern = frame['Environmental_Concern_Level']
    subsidy = frame['Subsidy_Available'].eq('Yes').astype(np.float32)
    home_charge = frame['Home_Charging_Possible'].eq('Yes').astype(np.float32)
    anxiety = frame['Range_Anxiety_Level'].map({'Low': 0.0, 'Medium': 1.0, 'High': 2.0}).fillna(1.0)

    # Known ordinal/binary meanings; nominal categories are one-hot encoded below.
    x['Subsidy_flag'] = subsidy
    x['Home_charging_flag'] = home_charge
    x['Range_anxiety_ordinal'] = anxiety.astype(np.float32)

    # Charging convenience and capacity interactions.
    total_stations = home_stations + work_stations
    x['Total_charging_stations'] = total_stations.astype(np.float32)
    x['Station_difference'] = (home_stations - work_stations).astype(np.float32)
    x['Station_minimum'] = np.minimum(home_stations, work_stations).astype(np.float32)
    x['Station_maximum'] = np.maximum(home_stations, work_stations).astype(np.float32)
    x['Station_product'] = (home_stations * work_stations).astype(np.float32)
    x['No_nearby_station'] = total_stations.eq(0).astype(np.float32)
    x['Either_station_zero'] = (home_stations.eq(0) | work_stations.eq(0)).astype(np.float32)
    x['Home_access_score'] = (home_stations + 3.0 * home_charge).astype(np.float32)
    x['Commute_per_station'] = safe_divide(commute, total_stations, offset=1.0).astype(np.float32)
    x['Stations_per_commute'] = safe_divide(total_stations, commute, offset=1.0).astype(np.float32)

    # Affordability / usage ratios and monotone companion transforms.
    x['Log_income'] = np.log1p(income).astype(np.float32)
    x['Log_commute'] = np.log1p(commute).astype(np.float32)
    x['Income_per_commute'] = safe_divide(income, commute, offset=1.0).astype(np.float32)
    x['Income_per_car'] = safe_divide(income, cars, offset=0.0).astype(np.float32)
    x['Income_x_subsidy'] = (income * subsidy).astype(np.float32)
    x['Concern_x_subsidy'] = (concern * subsidy).astype(np.float32)
    x['Concern_minus_anxiety'] = (concern - anxiety).astype(np.float32)
    x['Commute_x_anxiety'] = (commute * (anxiety + 1.0)).astype(np.float32)

    # The raw source-data formula gives trees a one-split summary and a strong prior.
    recipe_score = ev_recipe_score(frame)
    x['EV_recipe_score'] = recipe_score.astype(np.float32)
    x['EV_recipe_distance'] = (recipe_score - 5.5).astype(np.float32)
    x['EV_recipe_probability'] = ev_recipe_probability(frame).astype(np.float32)

    # Preserve raw tails. These companion views/flags expose clipping and hard edges.
    for col, (lower, upper) in clip_bounds.items():
        x[f'{col}_winsorized'] = frame[col].clip(lower, upper).astype(np.float32)
    x['Income_at_floor'] = income.eq(30_000).astype(np.float32)
    x['Commute_at_floor'] = commute.eq(5.0).astype(np.float32)
    x['Income_at_least_170k'] = income.ge(170_000).astype(np.float32)
    x['Commute_at_least_80'] = commute.ge(80.0).astype(np.float32)

    # Exact repeated values are a feature of this synthetic dataset, not measurement noise.
    for col in RAW_FEATURES:
        x[f'{col}_frequency'] = frame[col].map(frequency_maps[col]).fillna(0.0).astype(np.float32)

    # Digit decompositions expose modulo patterns that ordinary threshold splits cannot learn cheaply.
    categorical = frame[CATEGORICAL_COLS].copy()
    categorical['Subsidy_x_anxiety'] = frame['Subsidy_Available'] + '__' + frame['Range_Anxiety_Level']
    categorical['Subsidy_x_concern'] = frame['Subsidy_Available'] + '__' + concern.astype(int).astype(str)
    categorical['City_x_home_charge'] = frame['City_Type'] + '__' + frame['Home_Charging_Possible']
    categorical['City_x_car_type'] = frame['City_Type'] + '__' + frame['Current_Car_Type']

    income_integer = np.rint(income.to_numpy()).astype(np.int64)
    for place, label in ((1, 'ones'), (10, 'tens'), (100, 'hundreds'), (1_000, 'thousands'), (10_000, 'ten_thousands'), (100_000, 'hundred_thousands')):
        categorical[f'Income_digit_{label}'] = ((income_integer // place) % 10).astype(str)
    commute_tenths = np.rint(commute.to_numpy(dtype=np.float64) * 10.0).astype(np.int64)
    categorical['Commute_tenths_digit'] = (commute_tenths % 10).astype(str)
    categorical['Commute_ones_digit'] = ((commute_tenths // 10) % 10).astype(str)
    categorical['Commute_tens_digit'] = ((commute_tenths // 100) % 10).astype(str)
    age_integer = frame['Age'].to_numpy(dtype=np.int64)
    categorical['Age_ones_digit'] = (age_integer % 10).astype(str)

    one_hot = pd.get_dummies(categorical, prefix=categorical.columns, dtype=np.int8)
    x = pd.concat([x, one_hot], axis=1)
    x.columns = sanitize_feature_names(x.columns.astype(str).tolist())
    x = x.astype(np.float32)
    assert np.isfinite(x.to_numpy()).all(), 'Feature engineering produced NaN or infinity.'
    return x


all_base_features = build_base_features(all_raw)
BASE_FEATURE_NAMES = all_base_features.columns.tolist()
X_TRAIN_BASE = np.ascontiguousarray(all_base_features.iloc[: len(train)].to_numpy(dtype=np.float32))
X_TEST_BASE = np.ascontiguousarray(all_base_features.iloc[len(train):].to_numpy(dtype=np.float32))
del all_base_features, all_raw
gc.collect()

print(f'Base engineered matrix: {X_TRAIN_BASE.shape[1]} features')
print(f'Train matrix memory: {X_TRAIN_BASE.nbytes / 2**20:.1f} MiB')


In [ ]:
FEATURE_GROUPS = {
    'raw_numeric': list(NUMERIC_COLS),
    'binary_ordinal': ['Subsidy_flag', 'Home_charging_flag', 'Range_anxiety_ordinal'],
    'charging': [
        'Total_charging_stations', 'Station_difference', 'Station_minimum',
        'Station_maximum', 'Station_product', 'No_nearby_station',
        'Either_station_zero', 'Home_access_score',
        'Commute_per_station', 'Stations_per_commute',
    ],
    'ratios': ['Income_per_commute', 'Income_per_car'],
    'logs': ['Log_income', 'Log_commute'],
    'numeric_interactions': [
        'Income_x_subsidy', 'Concern_x_subsidy',
        'Concern_minus_anxiety', 'Commute_x_anxiety',
    ],
    'recipe_clues': ['EV_recipe_score', 'EV_recipe_distance', 'EV_recipe_probability'],
    'boundaries': [
        'Annual_Income_USD_winsorized', 'Daily_Commute_km_winsorized',
        'Income_at_floor', 'Commute_at_floor',
        'Income_at_least_170k', 'Commute_at_least_80',
    ],
    'frequency': [column for column in BASE_FEATURE_NAMES if column.endswith('_frequency')],
    'digits': [
        column for column in BASE_FEATURE_NAMES
        if column.startswith(('Income_digit_', 'Commute_tenths_digit_',
                              'Commute_ones_digit_', 'Commute_tens_digit_', 'Age_ones_digit_'))
    ],
    'category_interactions': [
        column for column in BASE_FEATURE_NAMES
        if column.startswith(('Subsidy_x_anxiety_', 'Subsidy_x_concern_',
                              'City_x_home_charge_', 'City_x_car_type_'))
    ],
}
assigned = [column for columns in FEATURE_GROUPS.values() for column in columns]
assert len(assigned) == len(set(assigned)), 'Feature groups overlap.'
assert set(assigned).issubset(BASE_FEATURE_NAMES), 'A registered column does not exist.'
FEATURE_GROUPS['raw_onehot'] = [column for column in BASE_FEATURE_NAMES if column not in set(assigned)]
assert all(column.startswith(tuple(name + '_' for name in CATEGORICAL_COLS))
           for column in FEATURE_GROUPS['raw_onehot'])
all_registered = [column for columns in FEATURE_GROUPS.values() for column in columns]
assert len(all_registered) == len(BASE_FEATURE_NAMES) == len(set(BASE_FEATURE_NAMES))
assert set(all_registered) == set(BASE_FEATURE_NAMES)
display(pd.DataFrame([{'group': name, 'n_features': len(columns)}
                      for name, columns in FEATURE_GROUPS.items()]))


In [ ]:
# name, grouping columns, smoothing strength, add recipe-residual encoding
TE_SPECS = [
    ('income_exact', ['Annual_Income_USD'], 80.0, True),
    ('commute_exact', ['Daily_Commute_km'], 150.0, True),
    ('income_subsidy_concern', ['Annual_Income_USD', 'Subsidy_Available', 'Environmental_Concern_Level'], 100.0, True),
    ('commute_anxiety', ['Daily_Commute_km', 'Range_Anxiety_Level'], 150.0, True),
    ('subsidy_concern_anxiety', ['Subsidy_Available', 'Environmental_Concern_Level', 'Range_Anxiety_Level'], 500.0, False),
    ('city_home', ['City_Type', 'Home_Charging_Possible'], 500.0, False),
    ('charger_pair', ['Charging_Stations_Near_Home', 'Charging_Stations_Near_Work'], 250.0, False),
    ('age_gender', ['Age', 'Gender'], 250.0, False),
]

TE_FEATURE_NAMES = []
for name, _, _, add_residual in TE_SPECS:
    TE_FEATURE_NAMES.append(f'TE_{name}')
    if add_residual:
        TE_FEATURE_NAMES.append(f'TE_recipe_residual_{name}')
ALL_FEATURE_NAMES = BASE_FEATURE_NAMES + TE_FEATURE_NAMES
RESIDUAL_TE_NAMES = [name for name in TE_FEATURE_NAMES if name.startswith('TE_recipe_residual_')]
STANDARD_TE_NAMES = [name for name in TE_FEATURE_NAMES if name not in RESIDUAL_TE_NAMES]


def smoothed_group_mean(
    fit_frame: pd.DataFrame,
    fit_values: np.ndarray,
    apply_frame: pd.DataFrame,
    columns: list[str],
    smoothing: float,
) -> np.ndarray:
    values = np.asarray(fit_values, dtype=np.float64)
    prior = float(values.mean())
    stats_frame = fit_frame[columns].copy()
    stats_frame['_encoding_value'] = values
    stats = stats_frame.groupby(columns, sort=False, dropna=False)['_encoding_value'].agg(['sum', 'count'])
    encoded_map = (stats['sum'] + smoothing * prior) / (stats['count'] + smoothing)

    if len(columns) == 1:
        encoded = apply_frame[columns[0]].map(encoded_map).to_numpy(dtype=np.float64)
    else:
        apply_index = pd.MultiIndex.from_frame(apply_frame[columns], names=columns)
        encoded = encoded_map.reindex(apply_index).to_numpy(dtype=np.float64)
    return np.nan_to_num(encoded, nan=prior).astype(np.float32)


def build_fold_target_encodings(
    fit_frame: pd.DataFrame,
    fit_y: np.ndarray,
    apply_frames: list[pd.DataFrame],
    seed: int,
    smoothing_multiplier: float = 1.0,
) -> tuple[np.ndarray, list[np.ndarray]]:
    assert smoothing_multiplier > 0
    fit_frame = fit_frame.reset_index(drop=True)
    apply_frames = [frame.reset_index(drop=True) for frame in apply_frames]
    fit_y = np.asarray(fit_y, dtype=np.float64)
    residual_y = fit_y - ev_recipe_probability(fit_frame)

    inner_cv = StratifiedKFold(n_splits=INNER_TE_FOLDS, shuffle=True, random_state=seed)
    inner_splits = list(inner_cv.split(np.zeros(len(fit_y)), fit_y.astype(np.uint8)))
    fit_columns: list[np.ndarray] = []
    apply_columns: list[list[np.ndarray]] = [[] for _ in apply_frames]

    for _, columns, smoothing, add_residual in TE_SPECS:
        smoothing = smoothing * smoothing_multiplier
        value_sets = [fit_y]
        if add_residual:
            value_sets.append(residual_y)

        for values in value_sets:
            cross_fitted = np.empty(len(fit_frame), dtype=np.float32)
            for inner_train_idx, inner_valid_idx in inner_splits:
                cross_fitted[inner_valid_idx] = smoothed_group_mean(
                    fit_frame.iloc[inner_train_idx],
                    values[inner_train_idx],
                    fit_frame.iloc[inner_valid_idx],
                    columns,
                    smoothing,
                )
            fit_columns.append(cross_fitted)

            for apply_number, apply_frame in enumerate(apply_frames):
                apply_columns[apply_number].append(
                    smoothed_group_mean(fit_frame, values, apply_frame, columns, smoothing)
                )

    fit_encoded = np.column_stack(fit_columns).astype(np.float32)
    apply_encoded = [np.column_stack(columns).astype(np.float32) for columns in apply_columns]
    assert fit_encoded.shape[1] == len(TE_FEATURE_NAMES)
    return fit_encoded, apply_encoded


print(f'Target-encoded features per fold: {len(TE_FEATURE_NAMES)}')
print(f'Total model features: {len(ALL_FEATURE_NAMES)}')


In [ ]:
assert FEATURE_GROUPS == confirmation_manifest['feature_groups']
assert TE_SPECS == [tuple(spec) for spec in confirmation_manifest['te_specs']]

DROPPED_FEATURES = {
    column
    for group in CANDIDATE_SPEC['drop_groups']
    for column in FEATURE_GROUPS[group]
}
assert len(DROPPED_FEATURES) == 12
BASE_KEEP_INDICES = np.array(
    [idx for idx, name in enumerate(BASE_FEATURE_NAMES) if name not in DROPPED_FEATURES],
    dtype=np.int64,
)
TE_KEEP_INDICES = np.arange(len(TE_FEATURE_NAMES), dtype=np.int64)
MODEL_FEATURE_NAMES = [BASE_FEATURE_NAMES[idx] for idx in BASE_KEEP_INDICES] + TE_FEATURE_NAMES

assert not DROPPED_FEATURES.intersection(MODEL_FEATURE_NAMES)
assert set(NUMERIC_COLS).issubset(MODEL_FEATURE_NAMES)
assert set(TE_FEATURE_NAMES).issubset(MODEL_FEATURE_NAMES)
assert len(MODEL_FEATURE_NAMES) == len(set(MODEL_FEATURE_NAMES))
assert len(MODEL_FEATURE_NAMES) == len(ALL_FEATURE_NAMES) - 12
assert len(MODEL_FEATURE_NAMES) == 193, 'Data vocabulary differs from the confirmed candidate.'

TEST_MARGIN = ev_recipe_logit_margin(test_raw)
assert np.isfinite(TEST_MARGIN).all()
print(f'Challenger model: {len(MODEL_FEATURE_NAMES)} features')
print('Excluded columns:', sorted(DROPPED_FEATURES))


In [ ]:
toy_fit = pd.DataFrame({'group': ['a', 'a', 'b', 'b'], 'level': [1, 1, 2, 2]})
toy_values = np.array([0.0, 1.0, 1.0, 1.0])
toy_apply = pd.DataFrame({'group': ['b', 'unknown', 'a'], 'level': [2, 99, 1]},
                         index=[100, 20, 5])
# Global mean = .75; with smoothing 2, a -> .625 and b -> .875.
expected = np.array([0.875, 0.75, 0.625], dtype=np.float32)
assert np.allclose(smoothed_group_mean(toy_fit, toy_values, toy_apply, ['group'], 2), expected)
assert np.allclose(smoothed_group_mean(toy_fit, toy_values, toy_apply, ['group', 'level'], 2), expected)

toy_size = 2 * INNER_TE_FOLDS
toy_raw = train_raw.iloc[:toy_size].reset_index(drop=True)
toy_y = np.tile(np.array([0, 1], dtype=np.uint8), INNER_TE_FOLDS)
toy_encoded, (toy_applied,) = build_fold_target_encodings(
    toy_raw, toy_y, [toy_raw.iloc[:3]], seed=GLOBAL_SEED
)
assert toy_encoded.shape == (toy_size, len(TE_FEATURE_NAMES))
assert toy_applied.shape == (3, len(TE_FEATURE_NAMES))
assert np.isfinite(toy_encoded).all() and np.isfinite(toy_applied).all()
assert len(ALL_FEATURE_NAMES) == len(set(ALL_FEATURE_NAMES))
assert all(not any(character in name for character in '[]<') for name in ALL_FEATURE_NAMES)
assert np.isfinite(X_TRAIN_BASE).all() and np.isfinite(X_TEST_BASE).all()
del toy_fit, toy_values, toy_apply, toy_raw, toy_y, toy_encoded, toy_applied
# Keep X_TEST_BASE and test_raw: every trained fold must predict the Kaggle test data.
# Raw train/test DataFrames are no longer needed after retaining IDs and cleaned features.
del train, test
gc.collect()
print('Preflight checks passed.')


## 3. Safe output directories and resumable training

Outputs belong only to `XGBoost/challenger/`. Each fingerprinted run gets its own manifest, feature list, checkpoints, models, metrics, OOF predictions, and test predictions.

A fold checkpoint is complete only after both validation and test predictions are available, and (when enabled) its model is saved. Only exact matching data/settings/checkpoints can resume. Old ablation checkpoints cannot supply test predictions and are not reused.

The final root-level submission CSV is written **only after all 20 fits are complete and validated**. A fingerprinted copy is also retained so a later run does not erase the current submission archive.


In [ ]:
def file_sha256(filename: Path) -> str:
    digest = hashlib.sha256()
    with filename.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json(value) -> str:
    return json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False)


def write_json_atomic(value, filename: Path) -> None:
    temporary = filename.with_name(filename.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
    temporary.replace(filename)


def write_csv_atomic(frame: pd.DataFrame, filename: Path) -> None:
    temporary = filename.with_name(filename.name + '.tmp')
    frame.to_csv(temporary, index=False, float_format='%.17g')
    temporary.replace(filename)


with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    saved_notebook = json.load(handle)
saved_code = '\n\n'.join(
    ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
    for cell in saved_notebook['cells'] if cell['cell_type'] == 'code'
)
data_hashes = {'train': file_sha256(TRAIN_PATH), 'test': file_sha256(TEST_PATH)}
assert data_hashes == confirmation_manifest['data_sha256'], 'Competition data changed since confirmation.'

RUN_CONFIG = {
    'pipeline_version': PIPELINE_VERSION, 'experiment': EXPERIMENT_NAME,
    'global_seed': GLOBAL_SEED, 'n_splits': N_SPLITS, 'split_seeds': list(SPLIT_SEEDS),
    'inner_te_folds': INNER_TE_FOLDS, 'max_boost_rounds': MAX_BOOST_ROUNDS,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS, 'max_bin': MAX_BIN,
    'device': DEVICE, 'nthread': NTHREAD, 'save_models': SAVE_MODELS,
    'versions': VERSIONS, 'reference_search_params': REFERENCE_SEARCH_PARAMS,
    'candidate_spec': CANDIDATE_SPEC, 'model_features': MODEL_FEATURE_NAMES,
    'feature_groups': FEATURE_GROUPS, 'te_specs': TE_SPECS,
    'data_sha256': {**data_hashes, 'sample_submission': file_sha256(SAMPLE_PATH)},
    'reference_configuration_sha256': file_sha256(CONFIG_PATH),
    'notebook_code_sha256': hashlib.sha256(saved_code.encode('utf-8')).hexdigest(),
    'model_seed_schedule': 'GLOBAL_SEED + 303 + 100000 * repeat_idx + fold_number',
    'encoding_seed_schedule': 'GLOBAL_SEED + 10000 + 100000 * repeat_idx + fold_number',
    'test_aggregation': 'equal probability average across folds, then equal average across split seeds',
}
FINGERPRINT = hashlib.sha256(canonical_json(RUN_CONFIG).encode('utf-8')).hexdigest()
RUN_DIR = STUDY_DIR / 'outputs' / FINGERPRINT[:16]
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
MODEL_DIR = RUN_DIR / 'models'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_MODELS:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = RUN_DIR / 'run_manifest.json'
if manifest_path.exists():
    with manifest_path.open(encoding='utf-8') as handle:
        if canonical_json(json.load(handle)) != canonical_json(RUN_CONFIG):
            raise ValueError('Existing run manifest differs; do not reuse its checkpoints.')
else:
    write_json_atomic(RUN_CONFIG, manifest_path)

write_json_atomic({
    'retained': MODEL_FEATURE_NAMES, 'removed': sorted(DROPPED_FEATURES),
    'base_indices': BASE_KEEP_INDICES.tolist(), 'te_indices': TE_KEEP_INDICES.tolist(),
}, RUN_DIR / 'experiment_features.json')
print('Run outputs:', RUN_DIR)
print('Final upload file will be:', STUDY_DIR / 'submission_xgboost_no_ratios_charging.csv')


In [ ]:
def model_params(model_seed: int) -> dict:
    params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc', 'booster': 'gbtree',
        'tree_method': 'hist', 'device': DEVICE, 'max_bin': MAX_BIN,
        'nthread': NTHREAD, 'seed': model_seed, 'verbosity': 0, 'validate_parameters': True,
        **REFERENCE_SEARCH_PARAMS,
    }
    if params['grow_policy'] == 'depthwise':
        params['max_leaves'] = 0
    else:
        params['max_depth'] = 0
    assert params['scale_pos_weight'] == 1.0
    return params


def prepare_matrix(base: np.ndarray, encoded: np.ndarray) -> np.ndarray:
    assert encoded.shape == (len(base), len(TE_FEATURE_NAMES))
    features = np.ascontiguousarray(
        np.column_stack([base[:, BASE_KEEP_INDICES], encoded[:, TE_KEEP_INDICES]]),
        dtype=np.float32,
    )
    assert features.shape[1] == len(MODEL_FEATURE_NAMES) and np.isfinite(features).all()
    return features


def validate_probabilities(predictions: np.ndarray, expected_rows: int, label: str) -> None:
    if predictions.shape != (expected_rows,) or not np.isfinite(predictions).all():
        raise ValueError(f'{label}: wrong shape or non-finite predictions.')
    if not np.all((predictions >= 0) & (predictions <= 1)):
        raise ValueError(f'{label}: predictions are outside [0, 1].')


def train_fold(
    train_indices: np.ndarray, valid_indices: np.ndarray,
    model_seed: int, encoding_seed: int, model_filename: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    fold_started = time.perf_counter()
    outer_train_raw = train_raw.iloc[train_indices]
    outer_valid_raw = train_raw.iloc[valid_indices]
    train_te, (valid_te, test_te) = build_fold_target_encodings(
        outer_train_raw, y[train_indices], [outer_valid_raw, test_raw],
        seed=encoding_seed, smoothing_multiplier=CANDIDATE_SPEC['smoothing_scale'],
    )
    train_features = prepare_matrix(X_TRAIN_BASE[train_indices], train_te)
    valid_features = prepare_matrix(X_TRAIN_BASE[valid_indices], valid_te)
    dtrain = xgb.QuantileDMatrix(
        train_features, label=y[train_indices], feature_names=MODEL_FEATURE_NAMES,
        max_bin=MAX_BIN, nthread=NTHREAD,
        base_margin=CANDIDATE_SPEC['prior_scale'] * ev_recipe_logit_margin(outer_train_raw),
    )
    dvalid = xgb.QuantileDMatrix(
        valid_features, label=y[valid_indices], feature_names=MODEL_FEATURE_NAMES,
        max_bin=MAX_BIN, nthread=NTHREAD, ref=dtrain,
        base_margin=CANDIDATE_SPEC['prior_scale'] * ev_recipe_logit_margin(outer_valid_raw),
    )
    del train_features, valid_features, train_te, valid_te, outer_train_raw, outer_valid_raw
    gc.collect()

    history = {}
    fit_started = time.perf_counter()
    model = xgb.train(
        params=model_params(model_seed), dtrain=dtrain, num_boost_round=MAX_BOOST_ROUNDS,
        evals=[(dvalid, 'valid')], evals_result=history, verbose_eval=False,
        callbacks=[xgb.callback.EarlyStopping(
            rounds=EARLY_STOPPING_ROUNDS, metric_name='auc', data_name='valid',
            maximize=True, save_best=True,
        )],
    )
    best_rounds = int(model.best_iteration) + 1
    valid_predictions = model.predict(dvalid, iteration_range=(0, best_rounds)).astype(np.float64)
    fit_seconds = float(time.perf_counter() - fit_started)
    validate_probabilities(valid_predictions, len(valid_indices), 'Validation')
    del dvalid
    gc.collect()

    # Build the test matrix after fitting; ref=dtrain ensures identical histogram cuts.
    test_started = time.perf_counter()
    test_features = prepare_matrix(X_TEST_BASE, test_te)
    dtest = xgb.QuantileDMatrix(
        test_features, feature_names=MODEL_FEATURE_NAMES,
        max_bin=MAX_BIN, nthread=NTHREAD, ref=dtrain,
        base_margin=CANDIDATE_SPEC['prior_scale'] * TEST_MARGIN,
    )
    del test_features, test_te
    test_predictions = model.predict(dtest, iteration_range=(0, best_rounds)).astype(np.float64)
    validate_probabilities(test_predictions, len(TEST_IDS), 'Test')
    test_prediction_seconds = float(time.perf_counter() - test_started)

    model_hash = None
    if SAVE_MODELS:
        temporary_model = model_filename.with_name(model_filename.stem + '.tmp.ubj')
        model.save_model(temporary_model)
        temporary_model.replace(model_filename)
        model_hash = file_sha256(model_filename)

    rounds_trained = len(history['valid']['auc'])
    metrics = {
        'auc': float(roc_auc_score(y[valid_indices], valid_predictions)),
        'best_rounds': best_rounds, 'rounds_trained': rounds_trained,
        'hit_round_limit': rounds_trained == MAX_BOOST_ROUNDS,
        'n_features': len(MODEL_FEATURE_NAMES), 'fit_seconds': fit_seconds,
        'test_prediction_seconds': test_prediction_seconds,
        'fold_total_seconds': float(time.perf_counter() - fold_started),
        'model_file': model_filename.name if SAVE_MODELS else None,
        'model_sha256': model_hash,
    }
    del model, dtrain, dtest
    gc.collect()
    return valid_predictions, test_predictions, metrics


def checkpoint_path(repeat_number: int, fold_number: int) -> Path:
    return CHECKPOINT_DIR / f'repeat_{repeat_number:02d}_fold_{fold_number:02d}.npz'


def model_path(repeat_number: int, fold_number: int) -> Path:
    return MODEL_DIR / f'repeat_{repeat_number:02d}_fold_{fold_number:02d}.ubj'


def save_fold_checkpoint(
    filename: Path, valid_indices: np.ndarray,
    valid_predictions: np.ndarray, test_predictions: np.ndarray, metrics: dict,
) -> None:
    temporary = filename.with_name(filename.name + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(
            handle, valid_indices=valid_indices, test_ids=TEST_IDS,
            valid_predictions=valid_predictions, test_predictions=test_predictions,
            fingerprint=np.array(FINGERPRINT), metrics_json=np.array(canonical_json(metrics)),
        )
    temporary.replace(filename)


def load_fold_checkpoint(
    filename: Path, valid_indices: np.ndarray, expected_model: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    with np.load(filename, allow_pickle=False) as saved:
        if str(saved['fingerprint'].item()) != FINGERPRINT:
            raise ValueError(f'Checkpoint configuration mismatch: {filename}')
        if not np.array_equal(saved['valid_indices'], valid_indices):
            raise ValueError(f'Checkpoint validation row mismatch: {filename}')
        if not np.array_equal(saved['test_ids'], TEST_IDS):
            raise ValueError(f'Checkpoint test row mismatch: {filename}')
        valid_predictions = saved['valid_predictions'].astype(np.float64)
        test_predictions = saved['test_predictions'].astype(np.float64)
        metrics = json.loads(str(saved['metrics_json'].item()))

    validate_probabilities(valid_predictions, len(valid_indices), 'Checkpoint validation')
    validate_probabilities(test_predictions, len(TEST_IDS), 'Checkpoint test')
    actual_auc = float(roc_auc_score(y[valid_indices], valid_predictions))
    if not np.isclose(actual_auc, metrics['auc'], atol=1e-12, rtol=0):
        raise ValueError(f'Checkpoint AUC mismatch: {filename}')
    if metrics['n_features'] != len(MODEL_FEATURE_NAMES):
        raise ValueError(f'Checkpoint feature count mismatch: {filename}')
    if not 0 < metrics['best_rounds'] <= metrics['rounds_trained'] <= MAX_BOOST_ROUNDS:
        raise ValueError(f'Checkpoint tree count mismatch: {filename}')
    if SAVE_MODELS:
        if metrics['model_file'] != expected_model.name or not expected_model.is_file():
            raise ValueError(f'Checkpoint saved model is missing: {expected_model}')
        if file_sha256(expected_model) != metrics['model_sha256']:
            raise ValueError(f'Checkpoint saved model checksum mismatch: {expected_model}')
    return valid_predictions, test_predictions, metrics


## 4. Train the challenger and predict the test rows

Only the challenger is trained. Folds, model seeds, and target-encoding seeds match the completed confirmation run exactly. Every fold checkpoint contains the additional test predictions that the old ablation checkpoints lack.

The loop skips already completed fits before constructing their large matrices. New fits retain only one model and its matrices at a time. No outcome from the test dataset or held-out validation fold is used to fit target encodings.


In [ ]:
n_repeats = len(SPLIT_SEEDS)
OOF_PREDICTIONS = np.full((n_repeats, len(y)), np.nan, dtype=np.float64)
TEST_PREDICTION_SUMS = np.zeros((n_repeats, len(TEST_IDS)), dtype=np.float64)
FOLD_ASSIGNMENTS = np.zeros((n_repeats, len(y)), dtype=np.uint8)
TEST_FOLD_COUNTS = np.zeros(n_repeats, dtype=np.int64)
FIT_RECORDS = []
planned_fits = n_repeats * N_SPLITS
completed_count = 0

for repeat_idx, split_seed in enumerate(SPLIT_SEEDS):
    repeat_number = repeat_idx + 1
    outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed)
    for fold_number, (train_indices, valid_indices) in enumerate(outer_cv.split(X_TRAIN_BASE, y), 1):
        FOLD_ASSIGNMENTS[repeat_idx, valid_indices] = fold_number
        model_seed = GLOBAL_SEED + 303 + 100_000 * repeat_idx + fold_number
        encoding_seed = GLOBAL_SEED + 10_000 + 100_000 * repeat_idx + fold_number
        filename = checkpoint_path(repeat_number, fold_number)
        model_filename = model_path(repeat_number, fold_number)
        loaded = RESUME_COMPLETED_FOLDS and filename.is_file()

        if loaded:
            valid_predictions, test_predictions, metrics = load_fold_checkpoint(
                filename, valid_indices, model_filename,
            )
        else:
            valid_predictions, test_predictions, metrics = train_fold(
                train_indices, valid_indices, model_seed, encoding_seed, model_filename,
            )
            save_fold_checkpoint(filename, valid_indices, valid_predictions, test_predictions, metrics)

        OOF_PREDICTIONS[repeat_idx, valid_indices] = valid_predictions
        TEST_PREDICTION_SUMS[repeat_idx] += test_predictions
        TEST_FOLD_COUNTS[repeat_idx] += 1
        FIT_RECORDS.append({
            'experiment': EXPERIMENT_NAME, 'repeat': repeat_number,
            'split_seed': split_seed, 'fold': fold_number,
            'model_seed': model_seed, 'encoding_seed': encoding_seed,
            'loaded_checkpoint': bool(loaded), **metrics,
        })
        completed_count += 1
        resumed = 'resumed' if loaded else 'trained'
        cap_note = ' [reached tree limit]' if metrics['hit_round_limit'] else ''
        print(f"[{completed_count}/{planned_fits}] repeat {repeat_number}, fold {fold_number}: "
              f"{resumed}, AUC={metrics['auc']:.8f}, best trees={metrics['best_rounds']}{cap_note}")
        write_csv_atomic(pd.DataFrame(FIT_RECORDS), RUN_DIR / 'fold_metrics.csv')
        del valid_predictions, test_predictions
        gc.collect()

    assert np.isfinite(OOF_PREDICTIONS[repeat_idx]).all()
    assert TEST_FOLD_COUNTS[repeat_idx] == N_SPLITS
    write_csv_atomic(pd.DataFrame({
        ID_COL: TRAIN_IDS, TARGET: y, 'fold': FOLD_ASSIGNMENTS[repeat_idx],
        'oof_no_ratios_charging': OOF_PREDICTIONS[repeat_idx],
    }), RUN_DIR / f'oof_repeat_{repeat_number:02d}.csv')
    print(f'Repeat {repeat_number} OOF AUC: '
          f'{roc_auc_score(y, OOF_PREDICTIONS[repeat_idx]):.10f}')

assert completed_count == planned_fits
assert np.all(FOLD_ASSIGNMENTS > 0)
assert np.isfinite(OOF_PREDICTIONS).all()
assert np.all(TEST_FOLD_COUNTS == N_SPLITS)
assert np.isfinite(TEST_PREDICTION_SUMS).all()
print('All folds have complete validation and test predictions.')


## 5. Validate and export the file to upload

The submission averages test **probabilities**, first across the ten folds of each split seed and then across both split seeds. It does not use ranks or optimize weights on the leaderboard.

Rows are aligned to `sample_submission.csv` by ID, and probabilities must be finite and within [0, 1]. Export is blocked until every required fold is complete. Both the fingerprinted submission and the convenient top-level copy are checked by reading them back with round-trip floating-point precision.

**Upload only `XGBoost/challenger/submission_xgboost_no_ratios_charging.csv`.** OOF CSVs, metrics, and test-prediction diagnostic CSVs are not submission files. The final cell prints its absolute path.


In [ ]:
fold_metrics = pd.DataFrame(FIT_RECORDS).sort_values(['repeat', 'fold']).reset_index(drop=True)
assert len(fold_metrics) == planned_fits
assert not fold_metrics.duplicated(['repeat', 'fold']).any()

TEST_PREDICTIONS_BY_REPEAT = TEST_PREDICTION_SUMS / TEST_FOLD_COUNTS[:, None]
FINAL_TEST_PREDICTIONS = TEST_PREDICTIONS_BY_REPEAT.mean(axis=0)
AVERAGED_OOF_PREDICTIONS = OOF_PREDICTIONS.mean(axis=0)
validate_probabilities(FINAL_TEST_PREDICTIONS, len(TEST_IDS), 'Final test')
validate_probabilities(AVERAGED_OOF_PREDICTIONS, len(TRAIN_IDS), 'Averaged OOF')

repeat_rows = []
for repeat_idx, split_seed in enumerate(SPLIT_SEEDS):
    repeat_rows.append({
        'experiment': EXPERIMENT_NAME, 'repeat': repeat_idx + 1, 'split_seed': split_seed,
        'oof_auc': float(roc_auc_score(y, OOF_PREDICTIONS[repeat_idx])),
    })
repeat_metrics = pd.DataFrame(repeat_rows)
averaged_oof_auc = float(roc_auc_score(y, AVERAGED_OOF_PREDICTIONS))
expected_oof_auc = frozen_configuration['confirmation_candidate_repeat_averaged_oof_auc']
print(f'Mean repeat OOF AUC: {repeat_metrics.oof_auc.mean():.10f}')
print(f'Two-seed averaged OOF AUC: {averaged_oof_auc:.10f}')
print(f'Completed confirmation candidate averaged OOF AUC: {expected_oof_auc:.10f}')
if not np.isclose(averaged_oof_auc, expected_oof_auc, atol=1e-7, rtol=0):
    print('WARNING: OOF score differs from confirmation; inspect settings/platform and metrics.')
if fold_metrics.hit_round_limit.any():
    print('WARNING: Some fits hit the tree limit; the frozen comparison budget is unchanged.')

write_csv_atomic(fold_metrics, RUN_DIR / 'fold_metrics.csv')
write_csv_atomic(repeat_metrics, RUN_DIR / 'repeat_oof_metrics.csv')
test_diagnostics = pd.DataFrame({ID_COL: TEST_IDS})
for repeat_idx in range(n_repeats):
    test_diagnostics[f'probability_seed_{repeat_idx + 1:02d}'] = TEST_PREDICTIONS_BY_REPEAT[repeat_idx]
test_diagnostics['probability_ensemble'] = FINAL_TEST_PREDICTIONS
write_csv_atomic(test_diagnostics, RUN_DIR / 'test_predictions.csv')
write_csv_atomic(pd.DataFrame({
    ID_COL: TRAIN_IDS, TARGET: y, 'oof_ensemble': AVERAGED_OOF_PREDICTIONS,
}), RUN_DIR / 'oof_ensemble.csv')

# Align by ID rather than assuming the sample file uses the test CSV's row order.
probability_by_id = pd.Series(FINAL_TEST_PREDICTIONS, index=TEST_IDS)
aligned_probabilities = probability_by_id.reindex(sample_submission[ID_COL]).to_numpy(dtype=np.float64)
validate_probabilities(aligned_probabilities, len(sample_submission), 'Submission')
submission = sample_submission.copy()
submission[TARGET] = aligned_probabilities
assert submission.columns.tolist() == [ID_COL, TARGET]
assert submission[ID_COL].is_unique and submission[TARGET].notna().all()

ARCHIVED_SUBMISSION_PATH = RUN_DIR / 'submission_xgboost_no_ratios_charging.csv'
SUBMISSION_PATH = STUDY_DIR / 'submission_xgboost_no_ratios_charging.csv'
write_csv_atomic(submission, ARCHIVED_SUBMISSION_PATH)
archived_check = pd.read_csv(ARCHIVED_SUBMISSION_PATH, float_precision='round_trip')
assert archived_check.columns.tolist() == [ID_COL, TARGET]
assert np.array_equal(archived_check[ID_COL].to_numpy(), sample_submission[ID_COL].to_numpy())
assert np.array_equal(archived_check[TARGET].to_numpy(), aligned_probabilities)
validate_probabilities(archived_check[TARGET].to_numpy(), len(TEST_IDS), 'Archived CSV')
write_csv_atomic(submission, SUBMISSION_PATH)
upload_check = pd.read_csv(SUBMISSION_PATH, float_precision='round_trip')
assert np.array_equal(upload_check[ID_COL].to_numpy(), archived_check[ID_COL].to_numpy())
assert np.array_equal(upload_check[TARGET].to_numpy(), archived_check[TARGET].to_numpy())

write_json_atomic({
    'fingerprint': FINGERPRINT, 'experiment': EXPERIMENT_NAME,
    'mean_repeat_oof_auc': float(repeat_metrics.oof_auc.mean()),
    'repeat_averaged_oof_auc': averaged_oof_auc,
    'expected_confirmation_repeat_averaged_oof_auc': expected_oof_auc,
    'completed_fits': completed_count, 'test_rows': len(TEST_IDS),
    'fits_reaching_round_limit': int(fold_metrics.hit_round_limit.sum()),
    'total_fit_minutes': float(fold_metrics.fit_seconds.sum() / 60),
    'total_test_prediction_minutes': float(fold_metrics.test_prediction_seconds.sum() / 60),
    'submission_sha256': file_sha256(SUBMISSION_PATH),
    'archived_submission': str(ARCHIVED_SUBMISSION_PATH.relative_to(PROJECT_ROOT)),
    'upload_file': str(SUBMISSION_PATH.relative_to(PROJECT_ROOT)),
    'note': 'Local OOF scores are not leaderboard scores; this is not the historical three-family rank blend.',
}, RUN_DIR / 'run_summary.json')

display(submission.head())
print(f'Validated submission: {len(submission):,} rows, columns {submission.columns.tolist()}')
print('UPLOAD THIS FILE:', SUBMISSION_PATH.resolve())
print('Archived copy:', ARCHIVED_SUBMISSION_PATH.resolve())
print('Your original model folders and ablation results were not modified.')
